In [1]:
!python baseline_3sigma.py

wrote C:\Users\tayyi\project\predictions_baseline.csv — 120 rows over 8 weeks


In [3]:
import os
os.listdir()

['.ipynb_checkpoints',
 'baseline_3sigma.py',
 'data',
 'predictions_baseline.csv',
 'Untitled.ipynb',
 'validate_submission.py']

In [9]:
!python baseline_3sigma.py --data data --out predictions.csv

wrote predictions.csv — 120 rows over 8 weeks


# Improvement of Code

In [98]:
import pandas as pd

# 1. LOAD DATA
df = pd.read_csv("predictions.csv")
meter = pd.read_csv("data/meter_read_success.csv")

# 2. CLEAN DATA
df["gateway_id"] = df["gateway_id"].str.strip()
meter["gateway_id"] = meter["gateway_id"].str.strip()

df["week_start"] = pd.to_datetime(df["week_start"])
meter["week_start"] = pd.to_datetime(meter["week_start"])

# 3. CREATE FAILURE RATE
meter["failure_rate"] = (
    (meter["meters_expected"] - meter["meters_read"])
    / meter["meters_expected"]
)

meter["failure_rate"] = meter["failure_rate"].fillna(0)

# 4. 🔥 IMPORTANT FIX
meter_latest = meter.sort_values("week_start").groupby("gateway_id").tail(1)


# 5. MERGE 
df = df.merge(
    meter_latest[["gateway_id", "failure_rate"]],
    on="gateway_id",
    how="left"
)

df["failure_rate"] = df["failure_rate"].fillna(0)

# 6. 🔥 STRONG IMPROVEMENT
# normalize baseline
df["base_norm"] = df["score"] / df["score"].max()

# final score
df["score"] = (
    df["base_norm"] * 380 +
    df["failure_rate"] * 600
)

# 7. RE-RANK (MANDATORY)
df = df.sort_values(["week_start", "score"], ascending=[True, False])
df["rank"] = df.groupby("week_start").cumcount() + 1

# keep only top 15
df = df[df["rank"] <= 15]

# 8. BETTER REASO
df["reason"] = df.apply(
    lambda x: f"{round(x.failure_rate*100,1)}% meter failure + anomaly signals instability",
    axis=1
)

# 9. FINAL FORMAT

df = df[["week_start", "rank", "gateway_id", "score", "reason"]]

# 10. SAVE
df.to_csv("predictions.csv", index=False)
print("✅ FINAL IMPROVEMENT DONE")

✅ FINAL IMPROVEMENT DONE


In [65]:
import shutil
shutil.copy("predictions.csv", "baseline_predictions.csv")

print("✅ Baseline saved")

✅ Baseline saved


In [90]:
!python validate_submission.py predictions.csv

predictions.csv: OK
  15 ranked gateways for each of 8 weeks, 2026-02-02 to 2026-03-23


# comparision 

In [100]:
old_df = pd.read_csv("baseline_predictions.csv")
new_df = pd.read_csv("predictions.csv")

print("🔴 BEFORE:")
print(old_df.head(10))

print("\n🟢 AFTER:")
print(new_df.head(10))

🔴 BEFORE:
   week_start  rank    gateway_id  score  \
0  2026-02-02     1  0A2778A31BE3   43.0   
1  2026-02-02     2  0E1B6F4DBA34   26.0   
2  2026-02-02     3  06F49BD8F572   26.0   
3  2026-02-02     4  064AED1EC0AA   26.0   
4  2026-02-02     5  0EF023E6452D   23.0   
5  2026-02-02     6  0A6324E3E043   22.0   
6  2026-02-02     7  0A4E33E2692B   22.0   
7  2026-02-02     8  02EBC6CD4398   20.0   
8  2026-02-02     9  0AA473D2FC17   20.0   
9  2026-02-02    10  06FFEC2EFA5E   19.0   

                                              reason  
0  43 hour(s) beyond 3 sigma of this gateway's ow...  
1  26 hour(s) beyond 3 sigma of this gateway's ow...  
2  26 hour(s) beyond 3 sigma of this gateway's ow...  
3  26 hour(s) beyond 3 sigma of this gateway's ow...  
4  23 hour(s) beyond 3 sigma of this gateway's ow...  
5  22 hour(s) beyond 3 sigma of this gateway's ow...  
6  22 hour(s) beyond 3 sigma of this gateway's ow...  
7  20 hour(s) beyond 3 sigma of this gateway's ow...  
8  20 hour

In [73]:
!python baseline_3sigma.py --data data --out predictions.csv

wrote predictions.csv — 120 rows over 8 weeks


# save files

In [76]:
baseline_df = pd.read_csv("predictions.csv")
baseline_df.to_csv("baseline_predictions.csv", index=False)